In [16]:
# path projet 
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SRC exists =", SRC_DIR.exists())

PROJECT_ROOT = /home/alouiyaz/projects/PINKCC/PINKCC_challenge_Cité_neutral_Minds
SRC exists = True


In [ ]:
# import
import matplotlib.pyplot as plt
from torch.utils.data import Subset, DataLoader

from data.dataset import BrainMRIDataset
from data.transforms import get_eval_transforms
from data.split import make_train_val_split
from models.resnet18 import build_resnet18
from pinkcc_ct_seg.training.engine import predict_probabilities
from pinkcc_ct_seg.utils import get_device, load_checkpoint, set_seed
from pinkcc_ct_seg.evaluation.thresholding import (
    apply_threshold,
    scan_thresholds,
    select_threshold_by_best_f1,
    select_threshold_by_recall_constraint,
)

In [18]:
# config 
set_seed(42)
device = get_device()

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

DATA_DIR = PROJECT_ROOT / "data/raw/brain_mri/Training"
TEST_DIR = PROJECT_ROOT / "data/raw/brain_mri/Testing"
MODEL_PATH = PROJECT_ROOT / "models/best_resnet18_methodB_advanced.pt"

print("device =", device)
print("MODEL_PATH exists =", MODEL_PATH.exists())

device = cpu
MODEL_PATH exists = True


In [19]:
# Dataset/loaders
eval_tfms = get_eval_transforms(img_size=IMG_SIZE)

full_train_ds = BrainMRIDataset(DATA_DIR, transform=eval_tfms)
labels = [full_train_ds.samples[i][1] for i in range(len(full_train_ds))]

train_idx, val_idx = make_train_val_split(labels, val_size=0.2, random_state=42)

val_ds = Subset(full_train_ds, val_idx)
test_ds = BrainMRIDataset(TEST_DIR, transform=eval_tfms)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print("Val size:", len(val_ds))
print("Test size:", len(test_ds))

Val size: 574
Test size: 394


In [20]:
# charger le modèle
model = build_resnet18(num_classes=2, pretrained=False)
model = load_checkpoint(model, MODEL_PATH, map_location=device)
model = model.to(device)
model.eval()

print("Model loaded.")

Model loaded.


In [21]:
# probabilités sur le set de validation
y_val, probs_val = predict_probabilities(model, val_loader, device)

print("Nb val targets:", len(y_val))
print("Nb val probs:", len(probs_val))
print("Exemple probs:", probs_val[:5])

Nb val targets: 574
Nb val probs: 574
Exemple probs: [0.9998099207878113, 0.8260790705680847, 0.9942591190338135, 0.9999592304229736, 0.9999991655349731]


In [ ]:
# probabilités sur le set de validation
y_val, probs_val = predict_probabilities(model, val_loader, device)

print("Nb val targets:", len(y_val))
print("Nb val probs:", len(probs_val))
print("Exemple probs:", probs_val[:5])


Nb val targets: 574
Nb val probs: 574
Exemple probs: [0.9998099207878113, 0.8260790705680847, 0.9942591190338135, 0.9999592304229736, 0.9999991655349731]


In [24]:
# Balayage des seuils
results = scan_thresholds(y_val, probs_val)
results[:5]

[{'threshold': 0.05,
  'accuracy': 0.975609756097561,
  'precision': 0.9762376237623762,
  'recall': 0.9959595959595959,
  'f1': 0.986},
 {'threshold': 0.1,
  'accuracy': 0.9825783972125436,
  'precision': 0.9859719438877755,
  'recall': 0.9939393939393939,
  'f1': 0.9899396378269618},
 {'threshold': 0.15000000000000002,
  'accuracy': 0.9790940766550522,
  'precision': 0.9859154929577465,
  'recall': 0.98989898989899,
  'f1': 0.9879032258064516},
 {'threshold': 0.2,
  'accuracy': 0.9825783972125436,
  'precision': 0.98989898989899,
  'recall': 0.98989898989899,
  'f1': 0.98989898989899},
 {'threshold': 0.25,
  'accuracy': 0.980836236933798,
  'precision': 0.9938775510204082,
  'recall': 0.9838383838383838,
  'f1': 0.9888324873096447}]

In [25]:
# meilleur seuil F1
best_f1_threshold = select_threshold_by_best_f1(results)
best_f1_threshold

{'threshold': 0.1,
 'accuracy': 0.9825783972125436,
 'precision': 0.9859719438877755,
 'recall': 0.9939393939393939,
 'f1': 0.9899396378269618}

In [26]:
# seuile avec recal sup 90%
best_recall90_threshold = select_threshold_by_recall_constraint(
    results,
    min_recall=0.90,
    min_precision=None
)

best_recall90_threshold

{'threshold': 0.1,
 'accuracy': 0.9825783972125436,
 'precision': 0.9859719438877755,
 'recall': 0.9939393939393939,
 'f1': 0.9899396378269618}

In [27]:
# seuil avec recall >=90% et precision >= 80%
best_clinical_threshold = select_threshold_by_recall_constraint(
    results,
    min_recall=0.90,
    min_precision=0.80
)

best_clinical_threshold

{'threshold': 0.1,
 'accuracy': 0.9825783972125436,
 'precision': 0.9859719438877755,
 'recall': 0.9939393939393939,
 'f1': 0.9899396378269618}

In [28]:
# courbe seuil /métriques
best_clinical_threshold = select_threshold_by_recall_constraint(
    results,
    min_recall=0.90,
    min_precision=0.80
)

best_clinical_threshold

{'threshold': 0.1,
 'accuracy': 0.9825783972125436,
 'precision': 0.9859719438877755,
 'recall': 0.9939393939393939,
 'f1': 0.9899396378269618}

In [31]:
# choisir un seuil finale
# variante F1 
final_threshold = best_f1_threshold["threshold"]
print("Seuil final (F1):", final_threshold)

# Variante métier 
final_threshold = best_clinical_threshold["threshold"]
print("Seuil final (métier):", final_threshold)


Seuil final (F1): 0.1
Seuil final (métier): 0.1


In [ ]:
# application au test finale
from sklearn.metrics import classification_report, confusion_matrix

y_test, probs_test = predict_probabilities(model, test_loader, device)
y_pred_test = apply_threshold(probs_test, final_threshold)


print("Threshold retenu:", final_threshold)
print(classification_report(y_test, y_pred_test, digits=4, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_test))

Threshold retenu: 0.1
              precision    recall  f1-score   support

           0     0.6757    0.9524    0.7905       105
           1     0.9797    0.8339    0.9009       289

    accuracy                         0.8655       394
   macro avg     0.8277    0.8931    0.8457       394
weighted avg     0.8987    0.8655    0.8715       394

Confusion matrix:
 [[100   5]
 [ 48 241]]


## Workflow 
1. entraînement du modèle
2. sauvegarde du meilleur modèle
3. calibration du seuil sur validation
4. choix du seuil selon une règle claire
5. test final avec ce seuil